- ## The problem of the long term Dependencies
- ## Stagnant Training
- ## Exploding Gradient Problem 
- we are not able to train the model completely

In [2]:
! pip install config

# The Math Behind Long-Term Dependencies in RNNs

## 1. Why This Problem Exists at All

A vanilla RNN processes a sequence by reusing the *same* weight matrices at every timestep, feeding the hidden state forward:

```
h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)
y_t = W_hy · h_t + b_y
```

Where:
- `h_t` = hidden state at time `t`
- `W_hh` = hidden-to-hidden (recurrent) weight matrix
- `W_xh` = input-to-hidden weight matrix
- `x_t` = input at time `t`

To learn a dependency between an event at timestep `t = 1` and an output at timestep `t = T` (say `T - 1 = 100` steps later), the network has to propagate an error signal backward through **all 100 intermediate steps** during training. This is called **Backpropagation Through Time (BPTT)**. The math of *why* this fails for long sequences is what "long-term dependency problem" refers to.

## 2. Setting Up Backpropagation Through Time

Suppose we have a loss `L` computed at the final timestep `T`, and we want the gradient of that loss with respect to the hidden state at some earlier timestep `k`:

```
∂L/∂h_k
```

By the chain rule, this gradient has to pass backward through every hidden state in between:

```
∂L/∂h_k = ∂L/∂h_T · ∂h_T/∂h_k
```

And `∂h_T/∂h_k` itself expands into a **product of Jacobians**, one for every timestep between `k` and `T`:

```
∂h_T/∂h_k = ∏_{t=k+1}^{T} (∂h_t/∂h_{t-1})
```

This product of matrices is the crux of the entire problem. Everything below is about what happens to a long product of matrices.

## 3. The Jacobian of a Single RNN Step

Differentiate the recurrence `h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)` with respect to `h_{t-1}`:

```
∂h_t/∂h_{t-1} = diag(tanh'(z_t)) · W_hh
```

where `z_t = W_hh · h_{t-1} + W_xh · x_t + b_h` is the pre-activation, and `tanh'(z) = 1 - tanh²(z)`.

Two things matter here:

1. **`tanh'(z)` is bounded**: its maximum value is `1`, achieved only at `z = 0`. For any nonzero activation, `tanh'(z) < 1`. In practice, once neurons saturate (large positive or negative `z`), `tanh'(z) → 0`.
2. **`W_hh` is a fixed matrix reused at every step.** Its eigenvalues (or more precisely, its singular values) determine whether repeated multiplication grows or shrinks a vector.

## 4. The Full Backward Product — Where It Breaks

Putting the single-step Jacobian into the multi-step chain rule:

```
∂h_T/∂h_k = ∏_{t=k+1}^{T} [diag(tanh'(z_t)) · W_hh]
```

This is a product of `(T - k)` matrices. To understand its magnitude, decompose `W_hh` using its eigendecomposition (assuming it's diagonalizable):

```
W_hh = Q · Λ · Q⁻¹
```

where `Λ = diag(λ_1, λ_2, ..., λ_n)` contains the eigenvalues. Then:

```
W_hh^n = Q · Λ^n · Q⁻¹
```

and each eigenvalue gets raised to the power `n`:

```
λ_i^n
```

**This is the entire problem in one line.** Whether the gradient explodes or vanishes as `n = T - k` grows depends entirely on the magnitude of `λ_i`:

| Condition | Behavior as `n → large` | Result |
|---|---|---|
| `|λ_i| < 1` | `λ_i^n → 0` | **Vanishing gradient** |
| `|λ_i| > 1` | `λ_i^n → ∞` | **Exploding gradient** |
| `|λ_i| = 1` | `λ_i^n` stays bounded | Stable (rare, unstable to maintain) |

Combined with the `tanh'(z_t) ≤ 1` factor at every step (which only ever *shrinks* things further, never grows them), the dominant real-world failure mode is **vanishing gradients** — exploding gradients happen too, but are comparatively easier to control (see §7).

## 5. A More Rigorous Bound

Using matrix norms instead of eigenvalues gives a cleaner inequality (this doesn't require `W_hh` to be diagonalizable, so it's the more general argument):

```
‖∂h_T/∂h_k‖ ≤ ∏_{t=k+1}^{T} ‖diag(tanh'(z_t))‖ · ‖W_hh‖
           ≤ (γ · ‖W_hh‖)^{T-k}
```

where `γ = max(tanh'(z)) = 1`. So the bound simplifies to:

```
‖∂h_T/∂h_k‖ ≤ ‖W_hh‖^{T-k}
```

This is the standard result quoted in Bengio, Simard & Frasconi (1994) and later Pascanu, Mikolov & Bengio (2013). It says: **the gradient norm is bounded by the spectral norm of `W_hh` raised to the power of the sequence length gap.** Any value other than exactly `‖W_hh‖ = 1` causes exponential growth or decay in `(T - k)`.

Since maintaining `‖W_hh‖ = 1` *exactly* through training via gradient descent is not something that happens naturally (weights get updated by gradients large numbers of times, drifting away from that knife-edge value), vanilla RNNs are essentially guaranteed to hit one of the two failure modes over long sequences.

## 6. Why This Specifically Kills *Long-Term* Dependencies

The exponent in `‖W_hh‖^{T-k}` is `T - k` — the **distance** between the timestep you're trying to learn from (`k`) and the timestep where the loss is computed (`T`). This has a direct consequence:

- For **short** dependencies (`T - k` small), the product has few terms, so it doesn't matter much whether `‖W_hh‖` is slightly above or below `1` — the gradient stays in a reasonable range.
- For **long** dependencies (`T - k` large, e.g., 50–100+ steps), even a mild deviation like `‖W_hh‖ = 0.9` gives:

```
0.9^100 ≈ 0.0000266
```

That gradient is numerically indistinguishable from zero by the time it reaches timestep `k`. The optimizer receives essentially no signal telling it "the input at timestep `k` mattered for the loss at timestep `T`," so the weights responsible for that early information never get updated to capture the dependency — **the network effectively cannot learn it.**

This is why the phenomenon is called *vanishing gradients causing long-term dependency loss*, not just "vanishing gradients" in isolation — the severity scales directly with the temporal gap.

## 7. The Exploding Case (Briefly)

If instead `‖W_hh‖ > 1`, the same product diverges:

```
1.1^100 ≈ 13,780
```

This causes huge, unstable weight updates — loss spikes to `NaN`, training diverges. This failure mode is easier to handle in practice via **gradient clipping**:

```
if ‖g‖ > threshold:
    g ← g · (threshold / ‖g‖)
```

This rescales the gradient vector's *direction* is preserved, but its magnitude is capped — a purely engineering fix, not a change to the underlying math of the Jacobian product. Vanishing gradients have no equivalent one-line fix, because you can't "clip upward" a signal that has genuinely lost its information (0.0000266 isn't recoverable — you don't know what it was supposed to be).

## 8. Why LSTMs/GRUs Fix This (The Math Difference)

The core architectural fix is replacing the multiplicative recurrence with an **additive** one. In an LSTM, the cell state update is:

```
c_t = f_t ⊙ c_{t-1} + i_t ⊙ c̃_t
```

Differentiate with respect to `c_{t-1}`:

```
∂c_t/∂c_{t-1} = f_t
```

(`⊙` is elementwise multiplication, `f_t` is the forget gate — a vector of values in `[0, 1]` produced by a sigmoid.) Contrast this with the vanilla RNN's Jacobian `diag(tanh'(z_t)) · W_hh`:

- The LSTM's gradient path is a simple elementwise multiplication by `f_t`, **not** a matrix multiplication by a fixed matrix raised to increasingly high powers.
- `f_t` is *learned per-timestep and per-input* by a gate, so the network can set `f_t ≈ 1` for information that needs to survive many steps unchanged, avoiding the forced exponential decay `‖W_hh‖^{T-k}` from §5 entirely.
- This additive path is often called the **"constant error carousel"** — gradients can flow through it across long spans without being repeatedly multiplied by the same shrinking (or exploding) matrix.

It doesn't make vanishing/exploding gradients *impossible* (if every forget gate outputs near-0, information still dies), but it makes the *default* behavior additive-preserving rather than multiplicative-decaying, which is why LSTMs handle dependencies over hundreds of timesteps far better than vanilla RNNs.

## 9. Summary

| Concept | Vanilla RNN | Why it matters |
|---|---|---|
| Recurrence | Multiplicative (`W_hh · h_{t-1}`, passed through `tanh`) | Forces repeated matrix multiplication |
| Backward gradient | `∏ diag(tanh'(z_t)) · W_hh` | Product of `(T-k)` Jacobians |
| Growth/decay rate | `‖W_hh‖^{T-k}` | Exponential in sequence gap |
| Failure mode | `‖W_hh‖ < 1` → vanish; `‖W_hh‖ > 1` → explode | Both are near-inevitable outcomes of training |
| Practical effect | Long-range dependencies (`T-k` large) get ~zero gradient signal | Network can't learn them |
| LSTM/GRU fix | Additive cell-state path (`f_t ⊙ c_{t-1}`) | Learned gate replaces forced exponential decay |

## References for Further Reading

- Bengio, Y., Simard, P., & Frasconi, P. (1994). *Learning long-term dependencies with gradient descent is difficult.* IEEE Transactions on Neural Networks.
- Pascanu, R., Mikolov, T., & Bengio, Y. (2013). *On the difficulty of training recurrent neural networks.* ICML.
- Hochreiter, S., & Schmidhuber, J. (1997). *Long Short-Term Memory.* Neural Computation.

### How to solve these Problems
- #### !st is Different Activation like Relu or Leaky Relu
- #### Better weight Initilization techniques
- #### Skip RNN
- #### LSTM

# Exploding Gradient Problem
- In this case if we will use the activation like relu or the value of lr is large in that case the multipluication will be more large and this will stop the trainnig as the values will depend more on the previous values

#### We can solve some of these problems by some of the ways like
- Gradient Clipping like in this we will restrict the value of gradient that it should not increase certain limit
- Controlled Learning Rate
